# Build a recording: tune every knob, then generate usable data

This is a **data-generation studio**, not a lesson. Where `01_anatomy` walks one
piece of physics at a time to *explain* it, here every knob is live at once so you
can *make a usable synthetic recording*. Three panels feed **one shared
configuration**:

1. **Anatomy & scope** - optics, sensor, tissue, cell placement, and the optical
   confounds (vasculature, illumination, vignette, glow, motion), against a live
   example frame. **Preset** buttons seed a real configuration to start from.
2. **Neural activity** - the calcium model, against live ground-truth traces.
3. **Generate** - set the length and write the recording to disk.

Run the cells top to bottom. Tune panel 1, then panel 2, then generate in panel 3;
each panel edits the same `config`, so the file you generate is exactly what you
previewed.

> The sliders need a **live kernel** - run this notebook, don't just read it. The
> previews run the real `minisim` forward model on a small, fast window, so what
> you tune is what the full-size file contains.

## Setup

The interactive backend builds one figure per panel and updates it in place as you drag a slider (smooth, and immune to VS Code's duplicate-output bug). We create the single `config` here; the three panels below all read and write it.

In [ ]:
%matplotlib widget
from IPython.display import display

from _studio_config import StudioConfig
from _studio_panels import AnatomyPanel, ActivityPanel, GeneratePanel

# The one shared configuration every panel tunes. Start from the library defaults;
# a preset button in panel 1 can replace these with a real scope/region in one click.
config = StudioConfig()

## 1. Anatomy & scope

Every non-activity knob, grouped in the accordion on the right, with two live views
on the left.

**Top: the example frame.** A **max-projection** of a short full-pipeline render
(the per-pixel brightest value over the window), so every cell that fires shows and
the per-frame shot noise averages out. To stay responsive, this is *not* the whole
field of view: it renders a centred window **auto-sized to a fixed cell budget**
(~350 cells) at the true pixel scale. As you raise the density the window shrinks to
hold that budget, so the *image's* apparent crowding stays roughly constant; the
optics, blur, noise and brightness are all at their real scale, so what you see is
faithful, just over a smaller patch. Auto-contrast keeps it visible at any exposure.

**Where to read the actual numbers.** The readout above the frame has two lines.
Line 1 is the *full-FOV* geometry and is the density-driven one: field of view,
pixel size, **expected neurons over the full FOV**, and the resolved focus depth.
Line 2 describes the preview window itself (its size, the cells shown, and how many
are detectable in this short clip; the full-length file recovers more). Because the
window auto-sizes to the cell budget, line 2's count stays near ~350 as you change
density - watch line 1 (and the side view) for the density.

**Bottom: the side view.** Where each layer sits in depth across the full FOV width:
the cell band (its dot count tracks the expected neuron count, so density is visible
here), the focal surface (a curve bowing shallower toward the edges when field
curvature is on), and the vasculature layer.

Click a **preset** to seed a real configuration (Generic, or Miniscope V4 imaging
CA1 or cortex L2/3), then fine-tune. Controls for a disabled stage (e.g. the vessel
sliders when vasculature is off) grey out.

> **Refresh time.** Adjusting a slider here takes a moment to redraw (a second or a
> few for a dense field) - this is the only panel with that lag. Each preview
> re-runs the optics, which recomputes the depth-dependent blur and dimming of
> *every cell's footprint*; that per-cell work dominates (and is why the preview
> renders a cell-budgeted window rather than the whole FOV). Sliders update on
> release, not continuously, so a drag triggers one redraw, not many. Panels 2 and 3
> are fast by comparison.

In [ ]:
anatomy = AnatomyPanel(config)
display(anatomy.ui())

## 2. Neural activity

The calcium model: a two-state firing gate (`burst onset/end p`, the in-burst and
baseline rates) feeding the indicator kinetics (`tau rise/decay`), with a
cell-to-cell brightness spread. The preview shows the **clean ground-truth** traces
`C` with spike ticks `S` for the busiest cells - measurement noise is a property of
the sensor and is added only in the generated file, never here.

`Quiet` / `Moderate` / `Active` set typical firing levels.

In [ ]:
activity = ActivityPanel(config)
display(activity.ui())

## 3. Generate the recording

Set the length, frame rate, and seed, choose where to write it, and generate at the
**full** field of view.

- **zarr** (default) writes the complete recording - the movie *and* the ground
  truth (footprints `A`, traces `C`, spikes `S`, positions, detectable flags, the
  vessel mask) plus the spec. This is what makes the data usable for testing an
  analysis pipeline. Reload it with `minisim.Recording.load(path)`.
- **avi** writes a viewable grayscale movie only (the spec is dropped beside it).

The size estimate flags how large the file (and, for zarr, the in-RAM movie) will
be - dial the duration down, or use avi, for a long recording.

In [ ]:
generate = GeneratePanel(config)
display(generate.ui())